# Rice seedlings detection and counting

This notebook use xxx_CNN model and compare with the classic CV baseline

# 1. Install, import packages and wandb setup

In [1]:
%pip install geoai-py ultralytics albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.8/601.8 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/13

In [2]:
import geoai
import geoai.train as gtrain
import rasterio
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import Polygon
import geopandas as gpd
import rasterio
import torch
from torch.utils.data import DataLoader
from ultralytics import YOLO
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from pathlib import Path
import xml.etree.ElementTree as ET
import json
import shutil
import glob
from tqdm import tqdm
import yaml
from PIL import Image

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
config = {
    "model_name": "yolo26m",
    "num_channels": 4,
    "num_classes": 2,
    "batch_size": 4,
    "num_epochs": 150,
    "learning_rate": 0.005,
    "tile_size": 512,
    "stride": 256,
    "val_split": 0.2,
    "project": "rice_seedlings_detection_counting",
    "name": "RSDC-YOLO26m"
}

# 2. Set up the folder and file paths

In [4]:
DATA_DIR = Path("/kaggle/input/datasets/ngtanphuc020505/rice-seedlings-dataset/riceseedling_dataset")

ANNOTATION_DIR = DATA_DIR / "annotations"
RAW_DIR = DATA_DIR / "raw"

# Create new directories
WORKING_DIR = Path("/kaggle/working")

DATASETS_DIR = WORKING_DIR / "datasets"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

IMG_DIR = DATASETS_DIR / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)
(IMG_DIR / "train").mkdir(parents=True, exist_ok=True)
(IMG_DIR / "val").mkdir(parents=True, exist_ok=True)
(IMG_DIR / "test").mkdir(parents=True, exist_ok=True)

LABELS_DIR = DATASETS_DIR / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
(LABELS_DIR / "train").mkdir(parents=True, exist_ok=True)
(LABELS_DIR / "val").mkdir(parents=True, exist_ok=True)
(LABELS_DIR / "test").mkdir(parents=True, exist_ok=True)

VECTOR_DIR = WORKING_DIR / "vectors"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

print("Raw images:", len(list(RAW_DIR.glob("*.tif"))))
print("XML labels:", len(list(ANNOTATION_DIR.glob("*.xml"))))

Raw images: 8
XML labels: 8


# 3. View the metadata of a raster
We need to get the basic information about what a raster might look like, what is Affine Transformation (so we can change the annotations from .xml to .geojson).

In [5]:
ANNO_2_PATH = RAW_DIR / "2.tif"

In [6]:
geoai.get_raster_info(ANNO_2_PATH)

{'driver': 'GTiff',
 'width': 1527,
 'height': 1527,
 'count': 4,
 'dtype': 'uint8',
 'crs': 'EPSG:3826',
 'transform': Affine(0.005239030778695925, 0.0, 218293.41517708264,
        0.0, -0.005239030779305829, 2658467.4298639484),
 'bounds': BoundingBox(left=218293.41517708264, bottom=2658459.4298639484, right=218301.4151770817, top=2658467.4298639484),
 'resolution': (0.01, 0.01),
 'nodata': None,
 'band_stats': [{'band': 1,
   'min': 33.0,
   'max': 249.0,
   'mean': 127.35750509600386,
   'std': 16.259917191192397},
  {'band': 2,
   'min': 36.0,
   'max': 254.0,
   'mean': 124.59662164857065,
   'std': 17.472386102087714},
  {'band': 3,
   'min': 46.0,
   'max': 230.0,
   'mean': 122.81249064535373,
   'std': 13.55456635998707},
  {'band': 4, 'min': 255.0, 'max': 255.0, 'mean': 255.0, 'std': 0.0}]}

The raster above has width x height x bands = 1527 x 1527 x 4. And the \
'transform': Affine(0.005239030778695925, 0.0, 218293.41517708264, 0.0, -0.005239030779305829, 2658467.4298639484) 

Affine matrix has 6 key parameters: [a, b, c, d, e, f]. Based on your data, these are:
- a: $0.005239$ (Pixel width)
- b: $0.0$ (Row rotation/skew)
- c: $218293.415$ (Top-Left X coordinate in meters)
- d: $0.0$ (Column rotation/skew)
- e: $-0.005239$ (Pixel height - negative because images draw top-to-bottom, but maps draw bottom-to-top)
- f: $2658467.429$ (Top-Left Y coordinate in meters)

Also, the top-left corner (c and f) of the image is:
- X (Easting) = 218293.415
- Y (Northing) = 2658467.429

Use the informations above to convert the vectors from .xml to .geojson

# 4. Change the .xml annotation to .txt for YOLO

We're going to use YOLOto train the model. This requires the annotations to be in .txt format. However, the annotations right now are in .xml format

This function reads a .xml file and returns a list of lists. Each inner list contains information of a bbox: xmin, ymin, xmax, ymax

In [7]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    bboxes = []

    for bbox in root.iter("bndbox"):
        info = []
        info.append(int(bbox.find("xmin").text))
        info.append(int(bbox.find("ymin").text))
        info.append(int(bbox.find("xmax").text))
        info.append(int(bbox.find("ymax").text))

        bboxes.append(info)

    return bboxes

This function is used when chipping the image. The purpose of this function is to clip the original box's coordinate relative to the tile

In [8]:
def clip_box_2_tile(bbox: list, tile_x, tile_y, tile_size):
    xmin, ymin, xmax, ymax = bbox

    # Calculate the new coordinate relative to the tile
    xmin -= tile_x
    ymin -= tile_y
    xmax -= tile_x
    ymax -= tile_y

    # Check if the bbox is in the tile
    if xmin <= 0 or ymin <= 0 or xmax >= tile_size or ymax >= tile_size:
        return None

    return [xmin, ymin, xmax, ymax]

Convert the .xml files to .txt

The bounding box annotation files required for training YOLO models must follow the format:

> File name  
Number of bounding boxes  
class x_center y_center width height

One row per object in the image.
One .txt file per image (with the same filename as the image).

The datasets needed to train YOLO should follow the folder tree structure as followed:

```
datasets
│
├── images
│   ├── train
│   └── val
│
└── labels
    ├── train
    └── val
```

In [9]:
TILE_SIZE = 640
STRIDE = 320

tile_metadata = {}  # {tile_id: {transform, crs, window}}
tile_count = 1
for img_path in tqdm(list(RAW_DIR.glob("*.tif"))):
    img_id = int(img_path.stem)
    xml_path = ANNOTATION_DIR / f"{img_id}.xml"

    src = rasterio.open(img_path, "r")
    img_width, img_height = src.width, src.height
    crs = str(src.crs)
    
    # Left-right, top-down
    for y in range(0, img_height, STRIDE):
        for x in range(0, img_width, STRIDE):
            # Define a window in the src, then create a tile (which is a new .tif file)
            window = Window(x, y, TILE_SIZE, TILE_SIZE)
            tile = src.read(window=window, boundless=True, fill_value=0)
            
            tile_file = f"{tile_count}.tif"
            
            # Create train:val:test data from images 1-5:6:7-8
            split = None
            if img_id <= 5:
                split = "train"
            elif img_id == 6:
                split = "val"        
            else:
                split = "test"
                
            # Save the tile

            # tile.shape = (4, 640, 640)
            # Band 1: Red
            # Band 2: Green  
            # Band 3: Blue
            # Band 4: NIR (Near-Infrared)
            rgb = tile[[0, 1, 2], :, :].astype(np.float32)  # take the RGB bands only
            rgb = rgb.astype(np.uint8).transpose(1, 2, 0)  # reshape from Channel, Width, Height to Width, Height, Channel
            rgb_img = Image.fromarray(rgb)  # create an image from an array
            rgb_img.save(IMG_DIR / split / f"{tile_count}.png")

            # Save geo metadata in .geojson format
            chip_transform = rasterio.windows.transform(window, src.transform)
            tile_metadata[tile_count] = {
                "source_image": img_id,
                "crs": crs,
                "col_off": x,
                "row_off": y,
                "transform": list(chip_transform)[:6]
            }
            
            # Create a label file for each tile
            label_file = f"{tile_count}.txt"
            label_path = LABELS_DIR / split / label_file

            # Get all the bboxes inside the tile
            img_bboxes = parse_xml(xml_path)
            yolo_lines = []
            for bbox in img_bboxes:
                clipped = clip_box_2_tile(bbox, x, y, TILE_SIZE)
                if clipped:
                    xmin, ymin, xmax, ymax = clipped

                    # Calculate and normalized
                    x_center = (xmin + xmax) / 2 / TILE_SIZE
                    y_center = (ymin + ymax) / 2 / TILE_SIZE
                    bbox_width = (xmax - xmin) / TILE_SIZE
                    bbox_height = (ymax - ymin) / TILE_SIZE

                    yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}")

            with open(label_path, "w") as f:
                f.write(f"\n".join(yolo_lines))

            tile_count += 1

# Save metadata of each tile
with open(VECTOR_DIR / "tile_metadata.json", "w") as f:
    json.dump(tile_metadata, f, indent=2)
    
print(f"✅ Tiling completed! Dataset saved at: {DATASETS_DIR}")

100%|██████████| 8/8 [00:39<00:00,  4.95s/it]

✅ Tiling completed! Dataset saved at: /kaggle/working/datasets


Create a data.yml

In [10]:
yaml_file_dict = {
    "path": "/kaggle/working/datasets",
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["rice_seedling"]
}

with open("/kaggle/working/datasets/data.yaml", "w") as f:
  yaml.dump(yaml_file_dict, f)

# 5. Train the object detection model

## Train object detection model

In [11]:
model = YOLO("yolo26m.pt")  # load the pretrained
model.info()  # print the model info

model.train(
    data="/kaggle/working/datasets/data.yaml",
    epochs=config["num_epochs"],
    imgsz=config["tile_size"],
    batch=config["batch_size"],
    workers=4,    
    patience=20,
    lr0=config["learning_rate"],     
    lrf=1e-2,     
    scale=0.3,          
    fliplr=0.35,
    flipud=0.4,
    hsv_s=0.4,    
    hsv_v=0.2,    
    mosaic=0.8,
    mixup=0.10,
    close_mosaic=10,
    shear=5,
    name=config["name"],
    project="/kaggle/working/rice_seedlings_detection_counting",
    device=0,
    save=True,
    save_period=-1
    )

YOLO26m summary: 280 layers, 21,896,248 parameters, 0 gradients, 75.4 GFLOPs
Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/datasets/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.35, flipud=0.4, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.2, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=RSDC-YOLO26m, nbs=64,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c73a5eebec0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

# 6. Evaluate the model

Create a test_data.yaml

In [12]:
test_yaml_file_dict = {
    "path": "/kaggle/working/datasets",
    "train": "images/train",
    "val": "images/test",
    "nc": 1,
    "names": ["rice_seedling"]
}

with open("/kaggle/working/datasets/test_data.yaml", "w") as f:
  yaml.dump(test_yaml_file_dict, f)

Evaluate the model

In [13]:
metrics = model.val(data="/kaggle/working/datasets/test_data.yaml")
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category
metrics.box.image_metrics  # per-image metrics dictionary with precision, recall, F1, TP, FP, and FN

Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2585.4±1076.8 MB/s, size: 491.8 KB)
val: Scanning /kaggle/working/datasets/labels/test... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 643.9it/s 0.1s
val: New cache created: /kaggle/working/datasets/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.9s/it 19.5s
                   all         50       6172      0.292      0.553      0.183      0.054
Speed: 3.8ms preprocess, 156.2ms inference, 2.8ms loss, 38.7ms postprocess per image
Results saved to /usr/local/lib/python3.12/dist-packages/tests/tmp/runs/detect/val


{'101.png': {'precision': 0.16333333333333333,
  'recall': 0.2737430167597765,
  'f1': 0.2045929018789144,
  'tp': 49,
  'fp': 251,
  'fn': 130},
 '102.png': {'precision': 0.16333333333333333,
  'recall': 0.28160919540229884,
  'f1': 0.20675105485232068,
  'tp': 49,
  'fp': 251,
  'fn': 125},
 '103.png': {'precision': 0.17,
  'recall': 0.28651685393258425,
  'f1': 0.21338912133891214,
  'tp': 51,
  'fp': 249,
  'fn': 127},
 '104.png': {'precision': 0.2833333333333333,
  'recall': 0.5182926829268293,
  'f1': 0.36637931034482757,
  'tp': 85,
  'fp': 215,
  'fn': 79},
 '105.png': {'precision': 0.21666666666666667,
  'recall': 0.9285714285714286,
  'f1': 0.35135135135135137,
  'tp': 65,
  'fp': 235,
  'fn': 5},
 '106.png': {'precision': 0.15333333333333332,
  'recall': 0.26136363636363635,
  'f1': 0.19327731092436973,
  'tp': 46,
  'fp': 254,
  'fn': 130},
 '107.png': {'precision': 0.15666666666666668,
  'recall': 0.27011494252873564,
  'f1': 0.19831223628691982,
  'tp': 47,
  'fp': 253,
 

In [14]:
results_path = "/kaggle/working/rice_seedlings_detection_counting/RSDC-YOLO26m/results.csv"
df = pd.read_csv(results_path)
df.columns = df.columns.str.strip()
print(df.to_string())

     epoch      time  train/box_loss  train/cls_loss  train/dfl_loss  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss    lr/pg0    lr/pg1    lr/pg2
0        1   24.6944         2.62943         3.02595         0.00473               0.58427            0.73867           0.60189              0.20504       2.20137       1.72536       0.00325  0.000620  0.000620  0.000620
1        2   31.1080         2.19756         1.37341         0.00347               0.50416            0.48970           0.33792              0.09426       3.01952       4.68674       0.00626  0.001252  0.001252  0.001252
2        3   37.4981         2.43757         1.24566         0.00397               0.60343            0.59196           0.47366              0.15883           NaN           NaN           NaN  0.001875  0.001875  0.001875
3        4   43.8286         2.25030         1.24267         0.00367               0.57014            0.48555       